In [1]:
import numpy as np

class DecisionStump:
    def __init__(self):
        self.feature_index = None
        self.threshold = None
        self.polarity = 1
        self.alpha = None  # learner weight

    def predict(self, X):
        n_samples = X.shape[0]
        predictions = np.ones(n_samples)

        if self.polarity == 1:
            predictions[X[:, self.feature_index] < self.threshold] = -1
        else:
            predictions[X[:, self.feature_index] > self.threshold] = -1

        return predictions


In [3]:
class AdaBoost:
    def __init__(self, n_estimators=10):
        self.n_estimators = n_estimators
        self.stumps = []

    def _bootstrap_samples(self, X, y):  # <<< CHANGED >>>
        n_samples = X.shape[0]
        indices = np.random.choice(n_samples, n_samples, replace=True)
        return X.iloc[indices], y.iloc[indices]

    def fit(self, X, y):
        n_samples, n_features = X.shape

        # Initialize weights uniformly
        w = np.ones(n_samples) / n_samples

        self.stumps = []

        for _ in range(self.n_estimators):
            X_boot, y_boot = self._bootstrap_samples(X, y)  # <<< CHANGED >>>
            Xb = X_boot.values                             # <<< CHANGED >>>
            yb = y_boot.values                             # <<< CHANGED >>>

            stump = DecisionStump()
            min_error = float('inf')

            for feature_i in range(n_features):
                thresholds = np.unique(Xb[:, feature_i])
                for threshold in thresholds:
                    for polarity in [1, -1]:
                        predictions = np.ones(len(yb))
                        if polarity == 1:
                            predictions[Xb[:, feature_i] < threshold] = -1
                        else:
                            predictions[Xb[:, feature_i] > threshold] = -1

                        error = np.sum(
                            w[X_boot.index][y_boot.values != predictions]
                        )  # <<< CHANGED >>>

                        if error < min_error:
                            min_error = error
                            stump.feature_index = feature_i
                            stump.threshold = threshold
                            stump.polarity = polarity

            # Compute alpha
            EPS = 1e-10
            stump.alpha = 0.5 * np.log((1 - min_error) / (min_error + EPS))

            full_predictions = stump.predict(X.values)  # <<< CHANGED >>>
            w *= np.exp(-stump.alpha * y.values * full_predictions)
            w /= np.sum(w)

            self.stumps.append(stump)

    def predict(self, X):
        X = X.values  # <<< CHANGED >>>
        stump_preds = np.array([
            stump.alpha * stump.predict(X)
            for stump in self.stumps
        ])
        return np.sign(np.sum(stump_preds, axis=0))

In [5]:
import pandas as pd
import numpy as np

dataset = pd.read_csv("/content/Heart Attack.csv")
dataset = dataset[['impluse', 'pressurehight', 'pressurelow', 'glucose', 'kcm', 'troponin', 'class']]
dataset['class'] = dataset['class'].map({'positive': 1, 'negative': -1})


train = dataset.sample(frac=0.8, random_state=0)
test = dataset.drop(train.index) # remaining 20%

X_ori = train.iloc[:, :-1]
X_ori_mean = X_ori.mean(axis=0)
X_ori_std  = X_ori.std(axis=0)
X_norm = (X_ori - X_ori_mean) / X_ori_std
y_target = train.iloc[:, -1]

X_train = X_norm
y_train = y_target
X_train = X_train.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)


X_test_ori = test.iloc[:, :-1]
y_test = test.iloc[:, -1]
X_test = (X_test_ori - X_ori_mean) / X_ori_std

X_test = X_test.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)


model = AdaBoost(n_estimators=10)
model.fit(X_train, y_train)

In [6]:
predictions = model.predict(X_test)
accuracy = np.sum(predictions == y_test) / len(y_test)
print(f"AdaBoost Accuracy: {accuracy * 100:.2f}%")

AdaBoost Accuracy: 96.59%
